# Eval-1000 users vs full-scale popularity baseline

This notebook checks whether the deterministic `eval_users1000` candidate budget gives stable popularity-baseline estimates across seeds, and how those estimates compare with the full-scale task.

The task construction is the same interaction-alignment setup used by the runner: full train split, capped Agent4Rec-style candidate groups, threshold selected on validation by macro-by-group F1, and test metrics reported both macro-by-group and micro.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def _add_repo_root_to_path() -> None:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and (path / "runners").exists():
            for import_path in (str(path), str(path / "src")):
                if import_path not in sys.path:
                    sys.path.insert(0, import_path)
            return
    raise RuntimeError("Could not find beyond-click-sim repo root")


_add_repo_root_to_path()

from runners.in_distribution.interaction_prediction.task_builders import (
    data_root,
    repo_root,
)

REPO_ROOT = repo_root()
DATA_ROOT = data_root()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
print(f"repo_root={REPO_ROOT}")
print(f"data_root={DATA_ROOT}")


In [ ]:
from time import perf_counter

from tqdm.auto import tqdm

from beyond_click_sim.evaluation import (
    apply_threshold,
    binary_classification_metrics,
    find_best_group_threshold,
    grouped_binary_classification_metrics,
)
from beyond_click_sim.scorers import PopularityScorer
from runners.in_distribution.interaction_prediction.methods.common import (
    candidate_group_summary,
    task_xy,
)
from runners.in_distribution.interaction_prediction.task_builders import (
    EVAL_USERS,
    NEGATIVE_RATIOS as RUNNER_NEGATIVE_RATIOS,
    SEEDS as RUNNER_SEEDS,
    build_alignment_task,
)


In [ ]:
DATASETS = ("ml-1m", "steam")

# Quick default: enough to inspect stability and the two ratio extremes.
# For the full Agent4Rec-style table, set NEGATIVE_RATIOS = (1, 2, 3, 9).
SEEDS = RUNNER_SEEDS
NEGATIVE_RATIOS = (1, 9)

# Full-scale is slower. We compare against seed 0 by default; extend if needed.
RUN_FULL_SCALE = True
FULL_SCALE_SEEDS = (0,)
FULL_SCALE_NEGATIVE_RATIOS = NEGATIVE_RATIOS

THRESHOLD_METRIC = "f1"
ITEM_COLUMN = "item_id"

print("runner seeds:", RUNNER_SEEDS)
print("runner ratios:", RUNNER_NEGATIVE_RATIOS)
print("eval users:", EVAL_USERS)


In [ ]:
def evaluate_popularity_task(task) -> dict[str, object]:
    xy = task_xy(task)
    X_train, y_train = xy["train"]
    X_val, y_val = xy["val"]
    X_test, y_test = xy["test"]

    candidate_group_column = task.schema.candidate_group_column
    if candidate_group_column is None:
        raise ValueError("Expected candidate_group_column in alignment task")

    scorer = PopularityScorer(item_column=ITEM_COLUMN).fit(X_train, y_train)
    val_scores = scorer.score(X_val)
    test_scores = scorer.score(X_test)

    threshold_selection = find_best_group_threshold(
        y_val,
        val_scores,
        X_val[candidate_group_column],
        metric=THRESHOLD_METRIC,
    )
    threshold = float(threshold_selection["threshold"])

    val_predictions = apply_threshold(val_scores, threshold)
    test_predictions = apply_threshold(test_scores, threshold)

    val_macro = grouped_binary_classification_metrics(
        y_val,
        val_predictions,
        X_val[candidate_group_column],
    )
    test_macro = grouped_binary_classification_metrics(
        y_test,
        test_predictions,
        X_test[candidate_group_column],
    )
    val_micro = binary_classification_metrics(y_val, val_predictions)
    test_micro = binary_classification_metrics(y_test, test_predictions)

    val_groups = candidate_group_summary(
        X_val,
        y_val,
        candidate_group_column=candidate_group_column,
        sampled_column=task.schema.sampled_column,
    )
    test_groups = candidate_group_summary(
        X_test,
        y_test,
        candidate_group_column=candidate_group_column,
        sampled_column=task.schema.sampled_column,
    )

    return {
        "task": task.name,
        "threshold": threshold,
        "threshold_metric_value": float(threshold_selection["metric_value"]),
        "train_rows": len(task.train),
        "val_rows": len(task.val),
        "test_rows": len(task.test),
        "test_users": int(X_test["user_id"].nunique()),
        "val_candidate_groups": val_groups["groups"],
        "test_candidate_groups": test_groups["groups"],
        "test_rows_mean": test_groups["rows_mean"],
        "test_rows_max": test_groups["rows_max"],
        "test_positives_mean": test_groups["positives_mean"],
        "val_macro_accuracy": val_macro["accuracy"],
        "val_macro_precision": val_macro["precision"],
        "val_macro_recall": val_macro["recall"],
        "val_macro_f1": val_macro["f1"],
        "test_macro_accuracy": test_macro["accuracy"],
        "test_macro_precision": test_macro["precision"],
        "test_macro_recall": test_macro["recall"],
        "test_macro_f1": test_macro["f1"],
        "test_micro_accuracy": test_micro["accuracy"],
        "test_micro_precision": test_micro["precision"],
        "test_micro_recall": test_micro["recall"],
        "test_micro_f1": test_micro["f1"],
    }


def run_popularity_spec(
    *,
    dataset_name: str,
    negative_ratio: int,
    seed: int,
    max_eval_users: int | None,
) -> dict[str, object]:
    scope = "full_scale" if max_eval_users is None else f"eval_users{max_eval_users}"

    build_start = perf_counter()
    task = build_alignment_task(
        dataset_name,
        negative_ratio,
        seed,
        max_eval_users=max_eval_users,
    )
    build_seconds = perf_counter() - build_start

    eval_start = perf_counter()
    result = evaluate_popularity_task(task)
    eval_seconds = perf_counter() - eval_start

    result.update(
        {
            "scope": scope,
            "dataset": dataset_name,
            "negative_ratio": negative_ratio,
            "seed": seed,
            "max_eval_users": max_eval_users,
            "build_seconds": round(build_seconds, 3),
            "eval_seconds": round(eval_seconds, 3),
        }
    )
    return result


def run_specs(specs: list[dict[str, object]]) -> pd.DataFrame:
    rows = []
    for spec in tqdm(specs, unit="task"):
        row = run_popularity_spec(**spec)
        rows.append(row)
        print(
            f"{row['scope']} {row['dataset']} m={row['negative_ratio']} "
            f"seed={row['seed']}: f1={row['test_macro_f1']:.4f}, "
            f"users={row['test_users']}, groups={row['test_candidate_groups']}, "
            f"build={row['build_seconds']}s"
        )
    return pd.DataFrame(rows)


In [ ]:
eval1000_specs = [
    {
        "dataset_name": dataset_name,
        "negative_ratio": negative_ratio,
        "seed": seed,
        "max_eval_users": EVAL_USERS,
    }
    for dataset_name in DATASETS
    for negative_ratio in NEGATIVE_RATIOS
    for seed in SEEDS
]

eval1000_results = run_specs(eval1000_specs)
eval1000_results.sort_values(["dataset", "negative_ratio", "seed"])


In [ ]:
eval1000_summary = (
    eval1000_results.groupby(["dataset", "negative_ratio"])
    .agg(
        seeds=("seed", "nunique"),
        test_macro_f1_mean=("test_macro_f1", "mean"),
        test_macro_f1_std=("test_macro_f1", "std"),
        test_macro_accuracy_mean=("test_macro_accuracy", "mean"),
        test_macro_precision_mean=("test_macro_precision", "mean"),
        test_macro_recall_mean=("test_macro_recall", "mean"),
        test_users_mean=("test_users", "mean"),
        test_candidate_groups_mean=("test_candidate_groups", "mean"),
        test_rows_mean=("test_rows", "mean"),
        build_seconds_mean=("build_seconds", "mean"),
    )
    .reset_index()
)
eval1000_summary

In [ ]:
full_scale_results = pd.DataFrame()

if RUN_FULL_SCALE:
    full_scale_specs = [
        {
            "dataset_name": dataset_name,
            "negative_ratio": negative_ratio,
            "seed": seed,
            "max_eval_users": None,
        }
        for dataset_name in DATASETS
        for negative_ratio in FULL_SCALE_NEGATIVE_RATIOS
        for seed in FULL_SCALE_SEEDS
    ]
    full_scale_results = run_specs(full_scale_specs)

full_scale_results.sort_values(["dataset", "negative_ratio", "seed"]) if not full_scale_results.empty else full_scale_results


In [ ]:
if full_scale_results.empty:
    comparison = pd.DataFrame()
    print("RUN_FULL_SCALE is False or no full-scale results were produced.")
else:
    eval_mean = (
        eval1000_results.groupby(["dataset", "negative_ratio"])
        .agg(
            eval1000_f1_mean=("test_macro_f1", "mean"),
            eval1000_f1_std=("test_macro_f1", "std"),
            eval1000_accuracy_mean=("test_macro_accuracy", "mean"),
            eval1000_precision_mean=("test_macro_precision", "mean"),
            eval1000_recall_mean=("test_macro_recall", "mean"),
            eval1000_groups_mean=("test_candidate_groups", "mean"),
            eval1000_rows_mean=("test_rows", "mean"),
        )
        .reset_index()
    )
    full_seed0 = full_scale_results[[
        "dataset",
        "negative_ratio",
        "seed",
        "test_macro_f1",
        "test_macro_accuracy",
        "test_macro_precision",
        "test_macro_recall",
        "test_candidate_groups",
        "test_rows",
    ]].rename(
        columns={
            "seed": "full_seed",
            "test_macro_f1": "full_f1",
            "test_macro_accuracy": "full_accuracy",
            "test_macro_precision": "full_precision",
            "test_macro_recall": "full_recall",
            "test_candidate_groups": "full_groups",
            "test_rows": "full_rows",
        }
    )

    comparison = eval_mean.merge(
        full_seed0,
        on=["dataset", "negative_ratio"],
        how="inner",
    )
    comparison["f1_delta_eval1000_minus_full"] = comparison["eval1000_f1_mean"] - comparison["full_f1"]
    comparison["accuracy_delta_eval1000_minus_full"] = comparison["eval1000_accuracy_mean"] - comparison["full_accuracy"]
    comparison = comparison.sort_values(["dataset", "negative_ratio"])

comparison


In [ ]:
# OUTPUT_DIR = REPO_ROOT / "outputs" / "in_distribution" / "interaction_prediction" / "notebook_eval1000_vs_full"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# eval1000_results.to_csv(OUTPUT_DIR / "eval1000_results.csv", index=False)
# eval1000_summary.to_csv(OUTPUT_DIR / "eval1000_summary.csv", index=False)
# if not full_scale_results.empty:
#     full_scale_results.to_csv(OUTPUT_DIR / "full_scale_results.csv", index=False)
# if not comparison.empty:
#     comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)

# print(f"saved to {OUTPUT_DIR}")
